# 03 - Apply Technical Constraints and Build Runs

This notebook:
1. Builds `manifests/runs.csv` from templates x counts x panels.
2. Materializes one constrained `.h5ad` per run in `data/runs/`.

Each run output is keyed by `run_id` and keeps the same schema expected by inferCNV + metrics notebooks.

In [1]:
from pathlib import Path
import itertools
import time
import numpy as np
import pandas as pd
import scanpy as sc

try:
    import yaml
except ImportError:
    yaml = None

In [2]:
ROOT = Path('/home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains')
CONFIG_PATH = ROOT / 'config' / 'technical_grid.yaml'
TEMPLATE_INDEX = ROOT / 'manifests' / 'template_adata_index.csv'
RUNS_MANIFEST = ROOT / 'manifests' / 'runs.csv'
RUNS_DIR = ROOT / 'data' / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

STATUS_LOG = ROOT / 'results' / 'logs' / 'run_status.csv'
STATUS_LOG.parent.mkdir(parents=True, exist_ok=True)

In [3]:
assert TEMPLATE_INDEX.exists(), f'Missing template index: {TEMPLATE_INDEX}'

if yaml is not None and CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)

    counts_fractions = cfg.get('counts_fractions', [100, 70, 50, 20, 10, 5, 3, 2, 1])
    gene_panels = cfg.get('gene_panels', ['all', 20000, 15000, 10000, 5000, 1000, 500])
    win_map = cfg.get('infercnv_window_sizes', {'all': 100, '20000': 80, '15000': 60, '10000': 40, '5000': 20, '1000': 4, '500': 2})
    seed_panel_base = cfg.get('seeds', {}).get('panel_sampling', 101)
    seed_counts_base = cfg.get('seeds', {}).get('count_sampling', 202)
else:
    counts_fractions = [100, 70, 50, 20, 10, 5, 3, 2, 1]
    gene_panels = ['all', 20000, 15000, 10000, 5000, 1000, 500]
    win_map = {'all': 100, '20000': 80, '15000': 60, '10000': 40, '5000': 20, '1000': 4, '500': 2}
    seed_panel_base = 101
    seed_counts_base = 202

counts_fractions = [int(x) for x in counts_fractions]
gene_panels = [str(x) for x in gene_panels]
win_map = {str(k): int(v) for k, v in win_map.items()}

counts_fractions, gene_panels, win_map

([100, 70, 50, 20, 10, 5, 3, 2, 1],
 ['all', '20000', '15000', '10000', '5000', '1000', '500'],
 {'all': 100,
  '20000': 80,
  '15000': 60,
  '10000': 40,
  '5000': 20,
  '1000': 4,
  '500': 2})

In [4]:
tpl_df = pd.read_csv(TEMPLATE_INDEX)
tpl_df[['template_id', 'output_h5ad', 'n_obs', 'n_vars']]

,template_id,output_h5ad,n_obs,n_vars
0,T01,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
1,T02,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
2,T03,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
3,T04,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
4,T05,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
5,T06,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
6,T07,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
7,T08,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
8,T09,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691
9,T10,/home/augusta/storage3/augusta/insituCNV/InSit...,1268,25691


## Build runs manifest

In [5]:
rows = []
for t in tpl_df.itertuples(index=False):
    t_id = t.template_id
    for c, g in itertools.product(counts_fractions, gene_panels):
        run_id = f'{t_id}_C{c}_G{g}'
        rows.append({
            'run_id': run_id,
            'template_id': t_id,
            'template_h5ad': t.output_h5ad,
            'count_fraction': int(c),
            'gene_panel': str(g),
            'window_size': int(win_map[str(g)]),
            'seed_panel': int(seed_panel_base),
            'seed_counts': int(seed_counts_base),
            'run_h5ad': str(RUNS_DIR / f'{run_id}.h5ad'),
            'status': 'pending',
        })

runs_df = pd.DataFrame(rows).sort_values(['template_id', 'count_fraction', 'gene_panel']).reset_index(drop=True)
runs_df.to_csv(RUNS_MANIFEST, index=False)
print(f'Saved runs manifest: {RUNS_MANIFEST}')
print(f'Total runs: {len(runs_df)}')
runs_df.head()

Saved runs manifest: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/manifests/runs.csv
Total runs: 630


,run_id,template_id,template_h5ad,count_fraction,gene_panel,window_size,seed_panel,seed_counts,run_h5ad,status
0,T01_C1_G1000,T01,/home/augusta/storage3/augusta/insituCNV/InSit...,1,1000,4,101,202,/home/augusta/storage3/augusta/insituCNV/InSit...,pending
1,T01_C1_G10000,T01,/home/augusta/storage3/augusta/insituCNV/InSit...,1,10000,40,101,202,/home/augusta/storage3/augusta/insituCNV/InSit...,pending
2,T01_C1_G15000,T01,/home/augusta/storage3/augusta/insituCNV/InSit...,1,15000,60,101,202,/home/augusta/storage3/augusta/insituCNV/InSit...,pending
3,T01_C1_G20000,T01,/home/augusta/storage3/augusta/insituCNV/InSit...,1,20000,80,101,202,/home/augusta/storage3/augusta/insituCNV/InSit...,pending
4,T01_C1_G500,T01,/home/augusta/storage3/augusta/insituCNV/InSit...,1,500,2,101,202,/home/augusta/storage3/augusta/insituCNV/InSit...,pending


## Materialize constrained run datasets
Creates one `.h5ad` per `run_id` in `data/runs/` by:
1) setting `X = layers['CNV_simulated']` from template adata,
2) downsampling counts globally,
3) re-normalizing/log1p in `CNV_simulated`,
4) subsampling genes to panel size.

In [6]:
OVERWRITE_EXISTING = False

def log_status(run_id, template_id, count_fraction, gene_panel, stage, status, message=''):
    now = pd.Timestamp.now()
    row = pd.DataFrame([{
        'run_id': run_id,
        'template_id': template_id,
        'count_fraction': count_fraction,
        'gene_panel': gene_panel,
        'stage': stage,
        'status': status,
        'start_time': now,
        'end_time': now,
        'duration_sec': 0.0,
        'message': message,
    }])
    if STATUS_LOG.exists():
        row.to_csv(STATUS_LOG, mode='a', header=False, index=False)
    else:
        row.to_csv(STATUS_LOG, index=False)

def apply_count_subsample(adata, fraction, seed):
    new = adata.copy()
    # Align with Figure2 approach: operate on CNV_simulated matrix in X
    new.X = new.layers['CNV_simulated'].copy()

    if fraction != 1.0:
        total_counts = int(fraction * new.X.sum())
        ds = sc.pp.downsample_counts(new, total_counts=total_counts, random_state=seed, copy=True)
        new.layers['CNV_simulated'] = ds.X.copy()
    else:
        new.layers['CNV_simulated'] = new.X.copy()

    new.layers['CNV_simulated_raw'] = new.layers['CNV_simulated'].copy()
    sc.pp.normalize_total(new, layer='CNV_simulated')
    sc.pp.log1p(new, layer='CNV_simulated')
    new.X = new.layers['CNV_simulated'].copy()
    return new

def apply_gene_panel(adata, panel_size, seed):
    if panel_size == 'all':
        return adata.copy()

    panel_n = int(panel_size)
    old_n = adata.n_vars
    if panel_n > old_n:
        raise ValueError(f'Panel size {panel_n} > n_vars {old_n}')

    rng = np.random.default_rng(seed)
    idx = rng.choice(old_n, size=panel_n, replace=False)
    idx = np.sort(idx)
    return adata[:, idx].copy()

In [7]:
runs_df = pd.read_csv(RUNS_MANIFEST)

# Cache templates in memory one by one to reduce reload overhead
for trow in tpl_df.itertuples(index=False):
    t_id = trow.template_id
    t_path = Path(trow.output_h5ad)
    print(f'\nLoading template {t_id}: {t_path}')
    adata_template = sc.read_h5ad(t_path)

    subset = runs_df[runs_df['template_id'] == t_id].copy()

    for r in subset.itertuples(index=False):
        run_h5ad = Path(r.run_h5ad)
        if run_h5ad.exists() and not OVERWRITE_EXISTING:
            continue

        t0 = time.time()
        try:
            frac = float(r.count_fraction) / 100.0
            ad_count = apply_count_subsample(adata_template, fraction=frac, seed=int(r.seed_counts))
            ad_run = apply_gene_panel(ad_count, panel_size=str(r.gene_panel), seed=int(r.seed_panel))

            # book-keeping
            ad_run.uns['run_id'] = r.run_id
            ad_run.uns['template_id'] = r.template_id
            ad_run.uns['count_fraction'] = int(r.count_fraction)
            ad_run.uns['gene_panel'] = str(r.gene_panel)
            ad_run.uns['window_size'] = int(r.window_size)

            ad_run.write_h5ad(run_h5ad, compression='gzip')
            dt = round(time.time() - t0, 2)
            log_status(r.run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'prepare_run', 'success', f'saved {run_h5ad.name} in {dt}s')
        except Exception as e:
            log_status(r.run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'prepare_run', 'failed', str(e))
            print(f'FAILED {r.run_id}: {e}')

print('Done materializing constrained runs.')


Loading template T01: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/data/intermediate/template_adatas/T01_fullgenes_simulated.h5ad

Loading template T02: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/data/intermediate/template_adatas/T02_fullgenes_simulated.h5ad

Loading template T03: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/data/intermediate/template_adatas/T03_fullgenes_simulated.h5ad

Loading template T04: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/data/intermediate/template_adatas/T04_fullgenes_simulated.h5ad

Loading template T05: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/data/intermediate/template_adatas/T05_fullgenes_simulated.h5ad

Loading template T06: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/data/intermediate/template_adatas/T06_fullgenes_simulated.h5a